# Structured Streaming using the Python DataFrames API

Apache Spark includes a high-level stream processing API, [Structured Streaming](http://spark.apache.org/docs/latest/structured-streaming-programming-guide.html). In this notebook we take a quick look at how to use the DataFrame API to build Structured Streaming applications. We want to compute real-time metrics like running counts and windowed counts on a stream of timestamped actions (e.g. Open, Close, etc).

To run this notebook, import it and attach it to a Spark cluster.

## Sample Data
We have some sample action data as files in `/databricks-datasets/structured-streaming/events/` which we are going to use to build this appication. Let's take a look at the contents of this directory.

In [0]:
# Look at the content of the following folder: /databricks-datasets/structured-streaming/events/
# What do you see?
import os 
FOLDER_PATH = r"/databricks-datasets/structured-streaming/events/" 
files = [ file for file in os.listdir(FOLDER_PATH)]
print(files)
# This folder contains json files.

['file-0.json', 'file-1.json', 'file-10.json', 'file-11.json', 'file-12.json', 'file-13.json', 'file-14.json', 'file-15.json', 'file-16.json', 'file-17.json', 'file-18.json', 'file-19.json', 'file-2.json', 'file-20.json', 'file-21.json', 'file-22.json', 'file-23.json', 'file-24.json', 'file-25.json', 'file-26.json', 'file-27.json', 'file-28.json', 'file-29.json', 'file-3.json', 'file-30.json', 'file-31.json', 'file-32.json', 'file-33.json', 'file-34.json', 'file-35.json', 'file-36.json', 'file-37.json', 'file-38.json', 'file-39.json', 'file-4.json', 'file-40.json', 'file-41.json', 'file-42.json', 'file-43.json', 'file-44.json', 'file-45.json', 'file-46.json', 'file-47.json', 'file-48.json', 'file-49.json', 'file-5.json', 'file-6.json', 'file-7.json', 'file-8.json', 'file-9.json']


There are about 50 JSON files in the directory. Let's see what each JSON file contains.

In [0]:
# Look at the functions head in dbutils
# Open one file

dbutils.fs.head( FOLDER_PATH+ files[0])

[Truncated to first 65536 bytes]


'{"time":1469501107,"action":"Open"}\n{"time":1469501147,"action":"Open"}\n{"time":1469501202,"action":"Open"}\n{"time":1469501219,"action":"Open"}\n{"time":1469501225,"action":"Open"}\n{"time":1469501234,"action":"Open"}\n{"time":1469501245,"action":"Open"}\n{"time":1469501246,"action":"Open"}\n{"time":1469501248,"action":"Open"}\n{"time":1469501256,"action":"Open"}\n{"time":1469501264,"action":"Open"}\n{"time":1469501266,"action":"Open"}\n{"time":1469501267,"action":"Open"}\n{"time":1469501269,"action":"Open"}\n{"time":1469501271,"action":"Open"}\n{"time":1469501282,"action":"Open"}\n{"time":1469501285,"action":"Open"}\n{"time":1469501291,"action":"Open"}\n{"time":1469501297,"action":"Open"}\n{"time":1469501303,"action":"Open"}\n{"time":1469501322,"action":"Open"}\n{"time":1469501335,"action":"Open"}\n{"time":1469501344,"action":"Open"}\n{"time":1469501346,"action":"Open"}\n{"time":1469501349,"action":"Open"}\n{"time":1469501357,"action":"Open"}\n{"time":1469501366,"action":"Open"}\n

Each line in the file contains JSON record with two fields - `time` and `action`. Let's try to analyze these files interactively.

## Batch/Interactive Processing
The usual first step in attempting to process the data is to interactively query the data. Let's define a static DataFrame on the files, and give it a table name.

In [0]:
from pyspark.sql.types import *

inputPath = "/databricks-datasets/structured-streaming/events/"

# Since we know the data format already, let's define the schema to speed up processing (no need for Spark to infer schema)
jsonSchema = StructType(
  [ StructField("time", TimestampType(), True),
   StructField("action", StringType(), True) ]
)



In [0]:
# Read all json files, taking into account the defined schema, and display the content 
staticDF = spark.read.schema(jsonSchema).json(inputPath)
staticDF.show()


+-------------------+------+
|               time|action|
+-------------------+------+
|2016-07-28 04:19:28| Close|
|2016-07-28 04:19:28| Close|
|2016-07-28 04:19:29|  Open|
|2016-07-28 04:19:31| Close|
|2016-07-28 04:19:31|  Open|
|2016-07-28 04:19:31|  Open|
|2016-07-28 04:19:32| Close|
|2016-07-28 04:19:33| Close|
|2016-07-28 04:19:35| Close|
|2016-07-28 04:19:36|  Open|
|2016-07-28 04:19:38| Close|
|2016-07-28 04:19:40|  Open|
|2016-07-28 04:19:41| Close|
|2016-07-28 04:19:42|  Open|
|2016-07-28 04:19:45|  Open|
|2016-07-28 04:19:47|  Open|
|2016-07-28 04:19:48|  Open|
|2016-07-28 04:19:49|  Open|
|2016-07-28 04:19:55|  Open|
|2016-07-28 04:20:00| Close|
+-------------------+------+
only showing top 20 rows


- Compare the dates from the output without schema and with it. 
- Did you notice that inputPath is a folder? Yes


In [0]:
staticDF_no_schema = spark.read.json(inputPath)
staticDF_no_schema.show()
# In the output without a schema, the values in the time column are timestamps.

+------+----------+
|action|      time|
+------+----------+
| Close|1469679568|
| Close|1469679568|
|  Open|1469679569|
| Close|1469679571|
|  Open|1469679571|
|  Open|1469679571|
| Close|1469679572|
| Close|1469679573|
| Close|1469679575|
|  Open|1469679576|
| Close|1469679578|
|  Open|1469679580|
| Close|1469679581|
|  Open|1469679582|
|  Open|1469679585|
|  Open|1469679587|
|  Open|1469679588|
|  Open|1469679589|
|  Open|1469679595|
| Close|1469679600|
+------+----------+
only showing top 20 rows


In [0]:
# Calculate the total number of 'Open' and 'Close' actions 
total_open_close = staticDF.groupBy("action").count()
total_open_close.show()


+------+-----+
|action|count|
+------+-----+
| Close|50000|
|  Open|50000|
+------+-----+



In [0]:
from pyspark.sql import functions as f
# Determine min and max time
min_max_time = staticDF.select(f.min("time"), f.max("time"))
min_max_time.show()



+-------------------+-------------------+
|          min(time)|          max(time)|
+-------------------+-------------------+
|2016-07-26 02:45:07|2016-07-28 06:48:19|
+-------------------+-------------------+



In [0]:
# Calculate the number of "open" and "close" actions with one hour windows: staticCountsDF
# Look at groupBy(..., window) function

staticCountsDF= staticDF.groupBy("action", f.window("time", "1 hour")).count()
staticCountsDF.show(truncate=False)
staticCountsDF.groupBy("action").sum("count").show()


+------+------------------------------------------+-----+
|action|window                                    |count|
+------+------------------------------------------+-----+
|Close |{2016-07-28 04:00:00, 2016-07-28 05:00:00}|960  |
|Open  |{2016-07-28 04:00:00, 2016-07-28 05:00:00}|825  |
|Close |{2016-07-27 13:00:00, 2016-07-27 14:00:00}|986  |
|Open  |{2016-07-26 13:00:00, 2016-07-26 14:00:00}|1006 |
|Close |{2016-07-26 13:00:00, 2016-07-26 14:00:00}|1028 |
|Close |{2016-07-26 14:00:00, 2016-07-26 15:00:00}|994  |
|Open  |{2016-07-27 04:00:00, 2016-07-27 05:00:00}|995  |
|Close |{2016-07-27 20:00:00, 2016-07-27 21:00:00}|1025 |
|Open  |{2016-07-27 20:00:00, 2016-07-27 21:00:00}|1005 |
|Close |{2016-07-27 21:00:00, 2016-07-27 22:00:00}|979  |
|Open  |{2016-07-27 23:00:00, 2016-07-28 00:00:00}|1008 |
|Close |{2016-07-27 23:00:00, 2016-07-28 00:00:00}|1011 |
|Close |{2016-07-28 00:00:00, 2016-07-28 01:00:00}|988  |
|Close |{2016-07-28 05:00:00, 2016-07-28 06:00:00}|671  |
|Close |{2016-

In [0]:
# Make this window a sliding window (30 minutes overlap): staticCountsSW
staticCountsDF= staticDF.groupBy("action", f.window("time", "1 hour", "30 minutes")).count()
staticCountsDF.show(truncate=False)
staticCountsDF.groupBy("action").sum("count").show()


+------+------------------------------------------+-----+
|action|window                                    |count|
+------+------------------------------------------+-----+
|Close |{2016-07-28 04:00:00, 2016-07-28 05:00:00}|960  |
|Open  |{2016-07-28 04:00:00, 2016-07-28 05:00:00}|825  |
|Close |{2016-07-28 04:30:00, 2016-07-28 05:30:00}|864  |
|Close |{2016-07-28 06:30:00, 2016-07-28 07:30:00}|33   |
|Open  |{2016-07-28 03:30:00, 2016-07-28 04:30:00}|989  |
|Close |{2016-07-27 12:30:00, 2016-07-27 13:30:00}|1034 |
|Close |{2016-07-27 13:00:00, 2016-07-27 14:00:00}|986  |
|Close |{2016-07-27 11:30:00, 2016-07-27 12:30:00}|989  |
|Open  |{2016-07-26 13:00:00, 2016-07-26 14:00:00}|1006 |
|Close |{2016-07-26 13:00:00, 2016-07-26 14:00:00}|1028 |
|Open  |{2016-07-26 13:30:00, 2016-07-26 14:30:00}|994  |
|Close |{2016-07-26 13:30:00, 2016-07-26 14:30:00}|1042 |
|Close |{2016-07-26 14:00:00, 2016-07-26 15:00:00}|994  |
|Close |{2016-07-26 12:30:00, 2016-07-26 13:30:00}|980  |
|Open  |{2016-

In [0]:
# Register staticCountsDF (createOrReplaceTempView) as table 'static_counts'
staticCountsDF.createOrReplaceTempView("static_counts")

Now we can directly use SQL to query the table.

In [0]:
%sql
-- Count all Open and Close actions in the table static_counts  
SELECT action, count(*) as count
FROM static_counts
GROUP BY action
    

action,count
Close,106
Open,102


In [0]:
%sql
-- How many actions (Close and Open separately) is within each time window (in the table static_counts)
-- Make a plot
SELECT * FROM static_counts

action,window,count
Close,"List(2016-07-28T04:00:00.000Z, 2016-07-28T05:00:00.000Z)",960
Open,"List(2016-07-28T04:00:00.000Z, 2016-07-28T05:00:00.000Z)",825
Close,"List(2016-07-28T04:30:00.000Z, 2016-07-28T05:30:00.000Z)",864
Close,"List(2016-07-28T06:30:00.000Z, 2016-07-28T07:30:00.000Z)",33
Open,"List(2016-07-28T03:30:00.000Z, 2016-07-28T04:30:00.000Z)",989
Close,"List(2016-07-27T12:30:00.000Z, 2016-07-27T13:30:00.000Z)",1034
Close,"List(2016-07-27T13:00:00.000Z, 2016-07-27T14:00:00.000Z)",986
Close,"List(2016-07-27T11:30:00.000Z, 2016-07-27T12:30:00.000Z)",989
Open,"List(2016-07-26T13:00:00.000Z, 2016-07-26T14:00:00.000Z)",1006
Close,"List(2016-07-26T13:00:00.000Z, 2016-07-26T14:00:00.000Z)",1028


Databricks visualization. Run in Databricks to view.

Note the two ends of the graph. The close actions are generated such that they are after the corresponding open actions, so there are more "opens" in the beginning and more "closes" in the end.

## Demo: Stream Processing 
Now that we have analyzed the data interactively, let's convert this to a streaming query that continuously updates as data comes. Since we just have a static set of files, we are going to emulate a stream from them by reading one file at a time, in the chronological order they were created. The query we have to write is pretty much the same as the interactive query above.

In [0]:
# Define the name of the new catalog
catalog = 'workspace'

# define variables for the trips data
schema = 'default'
volume = 'checkpoints'

# Path for file operations
path_volume = f'/Volumes/{catalog}/{schema}/{volume}'

# Three-part names for SQL operations
path_table = f'{catalog}.{schema}'
volume_name = f'{catalog}.{schema}.{volume}'

# Drop old temp volume (use three-part name, not path)
spark.sql(f"DROP VOLUME IF EXISTS {volume_name}")

# Create new temp volume
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")

# Define tmp dir for each stream
tmp_input = f"{path_volume}/input"
tmp_streaming_counts = f"{path_volume}/streaming_counts"
tmp_streaming_counts_filter = f"{path_volume}/streaming_counts_filter"
tmp_streaming_counts_run = f"{path_volume}/streaming_counts_run"


In [0]:
from pyspark.sql.functions import *

# Read data from a file
# Similar to definition of staticInputDF above, just using `readStream` instead of `read`
streamingInputDF = (
  spark
    .readStream                       
    .schema(jsonSchema)               # Set the schema of the JSON data
    .option("maxFilesPerTrigger", 1)  # Treat a sequence of files as a stream by picking one file at a time
    .json(inputPath)
)

# Do some transformations
# Same query as staticInputDF
streamingCountsDF = (                 
  streamingInputDF
    .groupBy(
      streamingInputDF.action, 
      window(streamingInputDF.time, "1 hour"))
    .count()
)

# Is this DF actually a streaming DF?
streamingCountsDF.isStreaming

True

In [0]:
# Display input data
streamingInputDF.display(checkpointLocation=tmp_input)

Checkpointing to /Volumes/workspace/default/checkpoints/input


In [0]:
# Display transformed data
streamingCountsDF.display(checkpointLocation=tmp_streaming_counts)

Checkpointing to /Volumes/workspace/default/checkpoints/streaming_counts


In [0]:
# Add aditional filter to transformed dataframe
streamingCountsDF.filter(streamingCountsDF.action == 'Open').display(checkpointLocation=tmp_streaming_counts_filter)

action,window,count
Open,"List(2016-07-27T01:00:00.000Z, 2016-07-27T02:00:00.000Z)",1004
Open,"List(2016-07-26T22:00:00.000Z, 2016-07-26T23:00:00.000Z)",997
Open,"List(2016-07-27T06:00:00.000Z, 2016-07-27T07:00:00.000Z)",1016
Open,"List(2016-07-26T02:00:00.000Z, 2016-07-26T03:00:00.000Z)",179
Open,"List(2016-07-26T13:00:00.000Z, 2016-07-26T14:00:00.000Z)",656
Open,"List(2016-07-27T14:00:00.000Z, 2016-07-27T15:00:00.000Z)",984
Open,"List(2016-07-27T11:00:00.000Z, 2016-07-27T12:00:00.000Z)",998
Open,"List(2016-07-26T07:00:00.000Z, 2016-07-26T08:00:00.000Z)",330
Open,"List(2016-07-26T23:00:00.000Z, 2016-07-27T00:00:00.000Z)",1000
Open,"List(2016-07-27T10:00:00.000Z, 2016-07-27T11:00:00.000Z)",1006


As you can see, `streamingCountsDF` is a streaming Dataframe (`streamingCountsDF.isStreaming` was `true`). You can start streaming computation, by defining the sink and starting it. 
In our case, we want to interactively query the counts (same queries as above), so we will set the complete set of 1 hour counts to be in a in-memory table.

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "2")  # keep the size of shuffles small

query = (
  streamingCountsDF
    .writeStream
    .format("memory")        # memory = store in-memory table 
    .queryName("counts")     # counts = name of the in-memory table
    .option("checkpointLocation", tmp_streaming_counts_run)
    .outputMode("complete")  # complete = all the counts should be in the table
    .trigger(availableNow=True)
    .start()
)

`query` is a handle to the streaming query that is running in the background. This query is continuously picking up files and updating the windowed counts. 

Note the status of query in the above cell. The progress bar shows that the query is active. 
Furthermore, if you expand the `> counts` above, you will find the number of files they have already processed. 

Let's wait a bit for a few files to be processed and then interactively query the in-memory `counts` table.

In [0]:
%sql
SELECT *
FROM counts

action,window,count


In [0]:
from time import sleep
sleep(5)  # wait a bit for computation to start

In [0]:
%sql
select action, date_format(window.end, "MMM-dd HH:mm") as time, count
from counts
order by time, action

action,time,count
Close,Jul-26 03:00,11
Open,Jul-26 03:00,179
Close,Jul-26 04:00,344
Open,Jul-26 04:00,1001
Close,Jul-26 05:00,815
Open,Jul-26 05:00,999
Close,Jul-26 06:00,323
Open,Jul-26 06:00,328
Close,Jul-26 14:00,699
Open,Jul-26 14:00,656


We see the timeline of windowed counts (similar to the static one earlier) building up. If we keep running this interactive query repeatedly, we will see the latest updated counts which the streaming query is updating in the background.

In [0]:
sleep(5)  # wait a bit more for more data to be computed

In [0]:
%sql select action, date_format(window.end, "MMM-dd HH:mm") as time, count from counts order by time, action

action,time,count
Close,Jul-26 03:00,11
Open,Jul-26 03:00,179
Close,Jul-26 04:00,344
Open,Jul-26 04:00,1001
Close,Jul-26 05:00,815
Open,Jul-26 05:00,999
Close,Jul-26 06:00,323
Open,Jul-26 06:00,328
Close,Jul-26 14:00,699
Open,Jul-26 14:00,656


In [0]:
sleep(5)  # wait a bit more for more data to be computed

In [0]:
%sql select action, date_format(window.end, "MMM-dd HH:mm") as time, count from counts order by time, action

action,time,count
Close,Jul-26 03:00,11
Open,Jul-26 03:00,179
Close,Jul-26 04:00,344
Open,Jul-26 04:00,1001
Close,Jul-26 05:00,815
Open,Jul-26 05:00,999
Close,Jul-26 06:00,1003
Open,Jul-26 06:00,1000
Close,Jul-26 07:00,328
Open,Jul-26 07:00,320


Also, let's see the total number of "opens" and "closes".

In [0]:
%sql 
select action, sum(count) as total_count 
from counts 
group by action 
order by action

action,total_count
Close,17529
Open,18471


If you keep running the above query repeatedly, you will always find that the number of "opens" is more than the number of "closes", as expected in a data stream where a "close" always appear after corresponding "open". This shows that Structured Streaming ensures **prefix integrity**. Read the blog posts linked below if you want to know more.

Note that there are only a few files, so consuming all of them there will be no updates to the counts. Rerun the query if you want to interact with the streaming query again.

Finally, you can stop the query running in the background, either by clicking on the 'Cancel' link in the cell of the query, or by executing `query.stop()`. Either way, when the query is stopped, the status of the corresponding cell above will automatically update to `TERMINATED`.

In [0]:
query.stop()

### IoT data

Develop a streaming example on `IoT device`dataset:

- inspect the dataset
- ask yourself couple of questions about the data and try to answer them (eg. how many steps users do, how many calories do they burn...)
- you read the data in streaming fashion (file by file) and keep the data for only one company? Here are some hints:
  - you can find the schema in the readme file 
  - as above, use this option: .option("maxFilesPerTrigger", 1)
  - use user_id or device_id for grouping
  - use timestamp for window definition
  - you can try streaming joins with the user data (/databricks-datasets/iot-stream/data-user/userData.csv). Here is the doc: https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#join-operations

In [0]:
# Iot stream dataset
display(dbutils.fs.ls('/databricks-datasets/iot-stream/'))

path,name,size,modificationTime
dbfs:/databricks-datasets/iot-stream/README.md,README.md,1596,1596565744000
dbfs:/databricks-datasets/iot-stream/data-device/,data-device/,0,1762699307200
dbfs:/databricks-datasets/iot-stream/data-user/,data-user/,0,1762699307200


In [0]:
# Read the README file
display(spark.read.text('/databricks-datasets/iot-stream/README.md'))

value
==========================================
IOT Device Data
==========================================
""
""
This dataset was created by Databricks.
It contains fake generated data in json and csv formats.
e.g.
"`{""user_id"": 12, ""calories_burnt"": 489.79998779296875, ""num_steps"": 9796, ""miles_walked"": 4.8979997634887695, ""time_stamp"": ""2018-07-24 03:54:00.893775"", ""device_id"": 10}`"
""


In [0]:
# Define the schema (copy from the README) data-device
schema = StructType([
  StructField("id",LongType(),False),  
  StructField("user_id",LongType(),True),  
  StructField("device_id",LongType(),True),  
  StructField("num_steps",LongType(),True),  
  StructField("miles_walked", FloatType(),True),  
  StructField("calories_burnt", FloatType(),True),  
  StructField("timestamp", StringType(), True),  
  StructField("value", StringType(), True)]  
)


user_schema = StructType([StructField("userid",IntegerType(),True),
 StructField("gender",StringType(),True),
 StructField("age",IntegerType(),True),
 StructField("height",IntegerType(),True),
 StructField("weight",IntegerType(),True),
 StructField("smoker",StringType(),True),
 StructField("familyhistory",StringType(),True),
 StructField("cholestlevs",StringType(),True),
 StructField("bp",StringType(),True),
 StructField("risk",IntegerType(),True)])

In [0]:
# Open one file to see how the data looks like (as a static dataframe)
data_device = "dbfs:/databricks-datasets/iot-stream/data-device/"
data_user = "dbfs:/databricks-datasets/iot-stream/data-user/"

staticUserDF = spark.read.schema(user_schema).csv(data_user, header=True).limit(1)
staticUserDF.show()

staticDeviceDF = spark.read.schema(schema).json(data_device).limit(1)
staticDeviceDF.show()



+------+------+---+------+------+------+-------------+-----------+----+----+
|userid|gender|age|height|weight|smoker|familyhistory|cholestlevs|  bp|risk|
+------+------+---+------+------+------+-------------+-----------+----+----+
|     1|     F| 40|    63|   100|     N|            Y|       High|High|   5|
+------+------+---+------+------+------+-------------+-----------+----+----+

+------+-------+---------+---------+------------+--------------+--------------------+--------------------+
|    id|user_id|device_id|num_steps|miles_walked|calories_burnt|           timestamp|               value|
+------+-------+---------+---------+------------+--------------+--------------------+--------------------+
|950000|     24|        5|     5014|       2.507|         250.7|2018-07-22 06:44:...|{"user_id": 24, "...|
+------+-------+---------+---------+------------+--------------+--------------------+--------------------+



## Devices

In [0]:
# Define your streaming dataframe

streamingInputDF = (
  spark
    .readStream                       
    .schema(schema)               # Set the schema of the JSON data
    .option("maxFilesPerTrigger", 1)  # Treat a sequence of files as a stream by picking one file at a time
    .json(data_device)
)

In [0]:
# Define your transformations

# Total number of steps, miles and calories burned per device
streamingCountsDF = (                 
  streamingInputDF
    .groupBy(
      streamingInputDF.device_id,
      window(streamingInputDF.timestamp, "10 seconds").alias("time_window"))
    .sum("num_steps", "miles_walked", "calories_burnt").withColumnRenamed("sum(num_steps)", "total_steps")           
    .withColumnRenamed("sum(miles_walked)", "total_miles") 
    .withColumnRenamed("sum(calories_burnt)", "total_calories") 
)

streamingCountsDF.isStreaming

True

In [0]:
# Define the sink and start streaming 
spark.conf.set("spark.sql.shuffle.partitions", "2")  # keep the size of shuffles small
tmp_streaming_sum = f"{path_volume}/streaming_sum"
query = (
  streamingCountsDF
    .writeStream
    .format("memory")        # memory = store in-memory table 
    .queryName("counts")     # counts = name of the in-memory table
    .option("checkpointLocation", tmp_streaming_sum)
    .outputMode("complete")  # complete = all the counts should be in the table
    .trigger(availableNow=True)
    .start()
)

In [0]:
%sql
-- visualize your streaming analytics

SELECT *
FROM counts

device_id,time_window,total_steps,total_miles,total_calories


In [0]:
sleep(5)

In [0]:
%sql
select *, date_format(time_window.end, "MMM-dd HH:mm") as time
from counts
order by time

device_id,time_window,total_steps,total_miles,total_calories,time
12,"List(2018-07-19T18:31:10.000Z, 2018-07-19T18:31:20.000Z)",10004,5.001999855041504,500.1999816894531,Jul-19 18:31
8,"List(2018-07-19T18:31:30.000Z, 2018-07-19T18:31:40.000Z)",11224,5.611999988555908,561.2000122070312,Jul-19 18:31
12,"List(2018-07-19T18:31:20.000Z, 2018-07-19T18:31:30.000Z)",7360,3.680000066757202,368.0,Jul-19 18:31
20,"List(2018-07-19T18:31:20.000Z, 2018-07-19T18:31:30.000Z)",5239,2.619499921798706,261.9499816894531,Jul-19 18:31
7,"List(2018-07-19T18:32:20.000Z, 2018-07-19T18:32:30.000Z)",9360,4.679999828338623,467.9999694824219,Jul-19 18:32
12,"List(2018-07-19T18:31:50.000Z, 2018-07-19T18:32:00.000Z)",10535,5.267499923706055,526.75,Jul-19 18:32
5,"List(2018-07-19T18:32:20.000Z, 2018-07-19T18:32:30.000Z)",6323,3.1614999771118164,316.1499938964844,Jul-19 18:32
13,"List(2018-07-19T18:32:00.000Z, 2018-07-19T18:32:10.000Z)",2021,1.0104999542236328,101.04999542236328,Jul-19 18:32
15,"List(2018-07-19T18:32:30.000Z, 2018-07-19T18:32:40.000Z)",6978,3.489000082015991,348.8999938964844,Jul-19 18:32
6,"List(2018-07-19T18:32:00.000Z, 2018-07-19T18:32:10.000Z)",7863,3.93149995803833,393.1499938964844,Jul-19 18:32


In [0]:
%sql
-- Number of steps, calories and miles per device
SELECT device_id, sum(total_steps) as total_steps, sum(total_calories) AS total_calories, sum(total_miles) as total_miles
FROM counts
GROUP BY device_id
ORDER BY device_id

device_id,total_steps,total_calories,total_miles
1,65204070,3260203.505470276,32602.035037636757
2,65867566,3293378.299972534,32933.78299796581
3,64846730,3242336.5041046143,32423.36502325535
4,65773606,3288680.295448303,32886.802980184555
5,62551708,3127585.402442932,31275.854009866714
6,64620400,3231020.0005493164,32310.199998140335
7,65146918,3257345.898536682,32573.458986401558
8,64774690,3238734.496734619,32387.34496331215
9,64642888,3232144.397491455,32321.44399678707
10,65229606,3261480.295715332,32614.80299282074


In [0]:
query.stop()